## Biomedical Named Entity Recognition with BERT
### 1. Project overview
Built a Biomedical Named Entity Recognition (NER) system using BERT fine-tuned on the BC5CDR dataset. The model identifies two biomedical entity types: Chemical and Disease.

In [ ]:
!pip install datasets transformers torch scikit-learn pandas matplotlib

In [2]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch

/Users/maryam/jupyter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Dataset & Preprocessing

The BC5CDR dataset provides word-level tokens and BIO labels. Since BERT uses subword tokenization, we aligned the original word level labels with BERT's subword tokens using "word_ids()".

For words split into multiple subwords, the first subword retained the original B- label while subsequent subwords received the corresponding I- label. Special tokens such as [CLS] and [SEP] were assigned -100 so they were ignored during training.

### 3. Model

Used pretrained bert-base-uncased with a token classification head containing 5 labels:

O, 
B-Chemical, 
I-Chemical, 
B-Disease, 
I-Disease

The pretrained BERT model was fine-tuned on the biomedical NER task.

In [3]:
dataset = load_dataset(
    "parquet",
    data_files={
        "train": "https://huggingface.co/datasets/tner/bc5cdr/resolve/refs%2Fconvert%2Fparquet/bc5cdr/train/0000.parquet",
        "validation": "https://huggingface.co/datasets/tner/bc5cdr/resolve/refs%2Fconvert%2Fparquet/bc5cdr/validation/0000.parquet",
        "test": "https://huggingface.co/datasets/tner/bc5cdr/resolve/refs%2Fconvert%2Fparquet/bc5cdr/test/0000.parquet"
    }
)

In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 5330
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 5865
    })
})


In [5]:
print(dataset["train"][0])

{'tokens': ['Naloxone', 'reverses', 'the', 'antihypertensive', 'effect', 'of', 'clonidine', '.'], 'tags': [1, 0, 0, 0, 0, 0, 1, 0]}


In [6]:
print(dataset["train"].features["tags"])

List(Value('int32'))


In [7]:
label_names = {
    0: "O",
    1: "B-Chemical",
    2: "B-Disease",
    3: "I-Disease",
    4: "I-Chemical"
}

In [8]:
tag_counts = {}

for example in dataset["train"]:
    for tag_id in example["tags"]:
        if tag_id in tag_counts:
            tag_counts[tag_id] += 1
        else:
            tag_counts[tag_id] = 1

for tag_id, count in sorted(tag_counts.items()):
    print("Tag ID: ", tag_id, "Count: ", count)


Tag ID:  0 Count:  96796
Tag ID:  1 Count:  5203
Tag ID:  2 Count:  4182
Tag ID:  3 Count:  2570
Tag ID:  4 Count:  571


In [9]:
for example in dataset["train"]:
    if 3 in example["tags"] or 4 in example["tags"]:
        for token, tag in zip(example["tokens"], example["tags"]):
            print(token, "->", label_names[tag])
        break

Lidocaine -> B-Chemical
- -> O
induced -> O
cardiac -> B-Disease
asystole -> I-Disease
. -> O


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [11]:
example = dataset["train"][0]

print(example["tokens"])
print(example["tags"])

['Naloxone', 'reverses', 'the', 'antihypertensive', 'effect', 'of', 'clonidine', '.']
[1, 0, 0, 0, 0, 0, 1, 0]


In [12]:
tokens = example["tokens"]

encoded = tokenizer(
    tokens,
    is_split_into_words=True
)

print(encoded.tokens())

['[CLS]', 'na', '##lo', '##xon', '##e', 'reverse', '##s', 'the', 'anti', '##hy', '##per', '##tens', '##ive', 'effect', 'of', 'cl', '##oni', '##dine', '.', '[SEP]']


In [13]:
word_ids = encoded.word_ids()

print(word_ids) #refer back to the original word's position

[None, 0, 0, 0, 0, 1, 1, 2, 3, 3, 3, 3, 3, 4, 5, 6, 6, 6, 7, None]


In [14]:
def tokenize_and_align_labels(example):
    
    tokens = example["tokens"]

    labels = example["tags"]

    encoded = tokenizer(
        tokens,
        is_split_into_words=True
    )

    word_ids = encoded.word_ids()

    aligned_labels = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
            # starting of subword
        elif word_id != previous_word_id:
            aligned_labels.append(labels[word_id])
        else:
            original_label = labels[word_id]

            if original_label == 1: # B-Chemical
                aligned_labels.append(4) # I-Chemical
            elif original_label == 2: # B-Disease
                aligned_labels.append(3) # I-Disease

            else:
                aligned_labels.append(original_label)
        previous_word_id = word_id

    encoded["labels"] = aligned_labels
        
    return encoded

In [15]:
# encoded, aligned_labels = align_labels(dataset["train"][0])

# print(encoded.tokens())
# print(aligned_labels)
'''
    ['[CLS]', 'na', '##lo', '##xon', '##e', 'reverse', '##s', 'the', 'anti', 
    '##hy', '##per', '##tens', '##ive', 'effect', 'of', 'cl', 
    '##oni', '##dine', '.', '[SEP]']
    [-100, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 0, -100]
'''

"\n    ['[CLS]', 'na', '##lo', '##xon', '##e', 'reverse', '##s', 'the', 'anti', \n    '##hy', '##per', '##tens', '##ive', 'effect', 'of', 'cl', \n    '##oni', '##dine', '.', '[SEP]']\n    [-100, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 0, -100]\n"

In [16]:
tokenized_train = dataset["train"].map(
    tokenize_and_align_labels
)

tokenized_validation = dataset["validation"].map(
    tokenize_and_align_labels
)

tokenized_test = dataset["test"].map(
    tokenize_and_align_labels
)

In [72]:
print(tokenized_train)
print(tokenized_validation)
print(tokenized_test)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5228
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5330
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5865
})


In [20]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5
)

Loading weights: 100%|████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 23075.88it/s]
[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignore

In [22]:
print(model.classifier)

Linear(in_features=768, out_features=5, bias=True)


Testing on an example:

In [23]:
example = tokenized_train[0]

input_ids = torch.tensor([example["input_ids"]])
attention_mask = torch.tensor([example["attention_mask"]])

In [24]:
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

In [25]:
print(outputs.logits.shape) 
# (batch_size, sequence_length, num_labels)
# 1 -> sentence, 20 -> BERT tokens, 5 -> possible NER labels

torch.Size([1, 20, 5])


In [26]:
predictions = torch.argmax(outputs.logits, dim=-1)
print(predictions) # returns index for each of the 20 BERT tokens

tensor([[3, 2, 2, 3, 2, 3, 2, 3, 3, 4, 4, 2, 3, 4, 2, 2, 2, 3, 3, 3]])


In [27]:
labels = torch.tensor([example["labels"]])

print(labels)

tensor([[-100,    1,    4,    4,    4,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    1,    4,    4,    0, -100]])


In [28]:
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels
)

print(outputs.loss)

tensor(1.7683, grad_fn=<NllLossBackward0>)


In [29]:
tokenized_train=tokenized_train.remove_columns(["tokens","tags"])
tokenized_validation=tokenized_validation.remove_columns(["tokens","tags"])
tokenized_test=tokenized_test.remove_columns(["tokens","tags"])

In [30]:
print(tokenized_train.column_names)

['input_ids', 'token_type_ids', 'attention_mask', 'labels']


### 4. Dynamic Padding

Different sentences have different numbers of tokens. A neural network batch requires tensors with compatible dimensions, so the shorter sequences need to be padded.

We used DataCollatorForTokenClassification to perform dynamic padding.

Instead of padding the entire dataset to one fixed maximum length, each batch is padded only to the length of its longest sequence.


In [31]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [32]:
batch = data_collator([
    tokenized_train[0],
    tokenized_train[1]
])

In [33]:
print(batch["attention_mask"].shape)

torch.Size([2, 51])


In [34]:
id2label = {
    0: "0",
    1: "B-Chemical",
    2: "B-Disease",
    3: "I-Disease",
    4: "I-Chemical"
}

label2id = {
    "0": 0,
    "B-Chemical": 1,
    "B-Disease": 2,
    "I-Disease": 3,
    "I-Chemical": 4
}

In [35]:
model.config.id2label = id2label
model.config.label2id = label2id

### 5. Training

The model was fine-tuned for 3 epochs using the Hugging Face Trainer. Validation performance was monitored after each epoch. The best model was retained using:
*load_best_model_at_end=True*

In [36]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

### 6. Evaluation
NER performance was evaluated using:

* Precision
* Recall
* F1 score
* Accuracy

seqeval was used because it is designed for sequence labeling tasks such as NER.

In [38]:
import evaluate

seqeval = evaluate.load("seqeval")

In [48]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions,labels):

        example_predictions = []
        example_labels = []

        for pred,lab in zip(prediction, label):

            if lab == -100:
                continue
            
            example_predictions.append(label_names[pred])
            example_labels.append(label_names[lab])

        true_predictions.append(example_predictions)
        true_labels.append(example_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [49]:
print(compute_metrics)

<function compute_metrics at 0x3147e2e80>


In [50]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [51]:
print(trainer.args.per_device_train_batch_size)

16


evaluating pretrained BERT before fine-tuning to establish baseline metrics

In [52]:
# evaluating pretrained BERT before fine-tuning 
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
No log,1.749231,0,0.002868,0.032426,0.005270,0.086150


{'eval_loss': 1.7492306232452393,
 'eval_precision': 0.0028683421720083007,
 'eval_recall': 0.03242623292670212,
 'eval_f1': 0.0052704718004338395,
 'eval_accuracy': 0.08615040195380075}

Evaluation of the pretrained BERT before fine-tuning gives us the following metrics:
* Precision: 0.28%
* Recall: 3.2%
* F1: 0.52%
* Accuracy is 8.6%



In [53]:
# training the model
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.127636,0.753603,0.855907,0.801504,0.955978
2,0.167922,0.122986,0.797535,0.863726,0.829312,0.961725
3,0.167922,0.137563,0.793663,0.867063,0.828741,0.960273


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.37it/s]
/Users/maryam/jupyter/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.24it/s]
/Users/maryam/jupyter/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.86it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.atte

TrainOutput(global_step=981, training_loss=0.11358975051739893, metrics={'train_runtime': 556.597, 'train_samples_per_second': 28.178, 'train_steps_per_second': 1.762, 'total_flos': 618932858551800.0, 'train_loss': 0.11358975051739893, 'epoch': 3.0})

In [54]:
print(trainer.state.log_history)

[{'eval_loss': 0.12763628363609314, 'eval_model_preparation_time': 0.0, 'eval_precision': 0.7536032314330304, 'eval_recall': 0.8559065790845585, 'eval_f1': 0.8015036125756688, 'eval_accuracy': 0.955978426783352, 'eval_runtime': 29.7088, 'eval_samples_per_second': 179.408, 'eval_steps_per_second': 11.242, 'epoch': 1.0, 'step': 327}, {'loss': 0.16792205810546876, 'grad_norm': 2.403320789337158, 'learning_rate': 9.826707441386342e-06, 'epoch': 1.529051987767584, 'step': 500}, {'eval_loss': 0.12298604846000671, 'eval_model_preparation_time': 0.0, 'eval_precision': 0.7975353807644171, 'eval_recall': 0.8637264101762069, 'eval_f1': 0.8293122434678146, 'eval_accuracy': 0.9617245005257623, 'eval_runtime': 34.36, 'eval_samples_per_second': 155.122, 'eval_steps_per_second': 9.721, 'epoch': 2.0, 'step': 654}, {'eval_loss': 0.13756296038627625, 'eval_model_preparation_time': 0.0, 'eval_precision': 0.7936629127696125, 'eval_recall': 0.8670628714419768, 'eval_f1': 0.8287408440878968, 'eval_accuracy':

In [56]:
test_results = trainer.evaluate(
    eval_dataset=tokenized_test
)

print(test_results)

/Users/maryam/jupyter/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.167922,0.128754,3,0.778264,0.852686,0.813777,0.959173


{'eval_loss': 0.1287543922662735, 'eval_precision': 0.7782637014980925, 'eval_recall': 0.8526863084922011, 'eval_f1': 0.8137769994162289, 'eval_accuracy': 0.9591730835757024}


After fine-tuning the model for 3 epochs, we achieved the following results:

* Precision: 77.82%
* Recall: 85.26%
* F1: 81.3%
* Accuracy is 95.91%

the F1 score considerably improved from the baseline F1 score of 0.53% after fine-tuning, thereby, demostrating that the pretrained BERT model has effectively adapted to the biomedical NER task.

### 7. Interactive Prediction
After training, an inference pipeline was created that allows users to enter their own medical sentences and receive detected entities.

Although BERT performs predictions at the subword level, the predictions are mapped back to the original words for readable output.


In [74]:
# post-processing, maps prediction back to original word
def predict_entities(text):
    
    words = text.split()

    encoded = tokenizer(
        words,
        is_split_into_words=True, # the things in words are already individual words. keep track of which BERT token came from which original word
        return_tensors="pt" # return pytorch tensors
    )

    device = trainer.model.device
    encoded = {key: value.to(device) for key,value in encoded.items()}

    with torch.no_grad():
        outputs = trainer.model(**encoded)


    predictions = torch.argmax(outputs.logits, dim=-1)[0]

    original_word_ids = tokenizer(
        words,
        is_split_into_words = True
    ).word_ids()

    word_predictions = []

    previous_word_id = None

    for prediction, word_id in zip(predictions.cpu().tolist(), original_word_ids):
        if word_id is None:
            continue

        if word_id != previous_word_id:
            word_predictions.append(
                (words[word_id], label_names[prediction])
            )

        previous_word_id = word_id

    return word_predictions

In [69]:
def extract_entities(text):
    predictions = predict_entities(text)

    for word, label in predictions:
        if label != "O":
            print(f"{word} -> {label}")

In [71]:
text=input("Enter a medical sentence: ")

extract_entities(text)

Enter a medical sentence:  she aspirin


aspirin -> I-Chemical


### 8. Conclusion
This is a complete biomedical NER pipeline using BERT, including preprocessing, subword label alignment, fine-tuning, evaluation, and interactive prediction. The resulting model can identify Chemical and Disease entities from biomedical text.

This provides a foundation for the next version, where the system will be expanded to recognize a wider range of clinical entities such as medications, symptoms, tests, and procedures.
